# 📊 OS Atrasadas por tipo de feriado
Análise de ordens de serviço atrasadas com identificação do tipo de feriados.

In [1]:
import pandas as pd
import numpy as np

print("✅ Bibliotecas carregadas com sucesso!")

✅ Bibliotecas carregadas com sucesso!


In [11]:
# Nomes padronizados das colunas
colunas_feriado = ['data', 'nome', 'tipo', 'descricao', 'uf', 'ibge']

# 1. Nacionais (Adicionado o caminho 'raw/')
nacionais = pd.read_csv('../raw/raw_feriado_nacional.csv', header=None, names=colunas_feriado)
nacionais['data'] = pd.to_datetime(nacionais['data'], format='%d/%m/%Y', errors='coerce')
datas_nac = set(nacionais['data'].dropna())

# 2. Estaduais (Adicionado o caminho 'raw/')
estaduais = pd.read_csv('../raw/raw_feriado_estadual.csv', header=None, names=colunas_feriado)
estaduais['data'] = pd.to_datetime(estaduais['data'], format='%d/%m/%Y', errors='coerce')
datas_est = set(estaduais['data'].dropna())

# 3. Municipais (Adicionado o caminho 'raw/')
municipais = pd.read_csv('../raw/raw_feriado_municipal.csv', header=None, names=colunas_feriado)
municipais['data'] = pd.to_datetime(municipais['data'], format='%d/%m/%Y', errors='coerce')
datas_mun = set(municipais['data'].dropna())

# 4. Facultativos (Adicionado o caminho 'raw/')
facultativos = pd.read_csv('../raw/raw_feriado_facultativo.csv', header=None, names=colunas_feriado)
facultativos['data'] = pd.to_datetime(facultativos['data'], format='%d/%m/%Y', errors='coerce')
datas_fac = set(facultativos['data'].dropna())

print(f"✅ Feriados carregados: {len(datas_nac)} Nacionais, {len(datas_est)} Estaduais, {len(datas_mun)} Municipais, {len(datas_fac)} Facultativos.")

✅ Feriados carregados: 9 Nacionais, 36 Estaduais, 359 Municipais, 15 Facultativos.


In [13]:
df_os = pd.read_csv('../refined/grafana/os_data.csv') 

# Converter colunas de data para datetime
df_os['data_entrada_efetiva'] = pd.to_datetime(df_os['data_entrada_efetiva'], errors='coerce')
df_os['data_saida_efetiva'] = pd.to_datetime(df_os['data_saida_efetiva'], errors='coerce')

# Filtrar apenas OS finalizadas e com datas válidas
df_os = df_os[df_os['status'] == 'FINALIZADO'].dropna(subset=['data_entrada_efetiva', 'data_saida_efetiva'])

# Calcular o tempo efetivo de entrega (em dias)
df_os['tempo_entrega_dias'] = (df_os['data_saida_efetiva'] - df_os['data_entrada_efetiva']).dt.days

# Tratar possíveis inconsistências (valores negativos de dias)
df_os = df_os[df_os['tempo_entrega_dias'] >= 0]

print(f"✅ {len(df_os)} Ordens de Serviço prontas para análise.")

✅ 51 Ordens de Serviço prontas para análise.


In [14]:
def checar_impacto(row, feriados_set):
    # Se a lista de datas do feriado estiver vazia, retorna Falso
    if not feriados_set:
        return False
        
    periodo_aberto = pd.date_range(start=row['data_entrada_efetiva'], end=row['data_saida_efetiva'])
    if any(data in feriados_set for data in periodo_aberto):
        return True
        
    janela_pos_feriado = pd.date_range(end=row['data_saida_efetiva'], periods=4)[:-1]
    if any(data in feriados_set for data in janela_pos_feriado):
        return True
        
    return False

# Aplicar verificação independente para cada tipo de feriado
df_os['impacto_nacional'] = df_os.apply(lambda r: checar_impacto(r, datas_nac), axis=1)
df_os['impacto_estadual'] = df_os.apply(lambda r: checar_impacto(r, datas_est), axis=1)
df_os['impacto_municipal'] = df_os.apply(lambda r: checar_impacto(r, datas_mun), axis=1)
df_os['impacto_facultativo'] = df_os.apply(lambda r: checar_impacto(r, datas_fac), axis=1)

# Impacto Geral (se a OS sofreu impacto de QUALQUER UM dos tipos acima)
df_os['impactada_por_feriado'] = (
    df_os['impacto_nacional'] | 
    df_os['impacto_estadual'] | 
    df_os['impacto_municipal'] | 
    df_os['impacto_facultativo']
)

print("✅ Análise de impacto concluída e categorizada por tipo!")

✅ Análise de impacto concluída e categorizada por tipo!


In [21]:
# Extrair o ano e o mês separadamente com base na data de saída
df_os['ano'] = df_os['data_saida_efetiva'].dt.year
df_os['mes'] = df_os['data_saida_efetiva'].dt.month

# Calcular a média de tempo de entrega agrupada por Ano e Mês para cada tipo de feriado
df_nac = df_os[df_os['impacto_nacional'] == True].groupby(['ano', 'mes'])['tempo_entrega_dias'].mean().to_frame('tmp_feriado').reset_index()
df_nac['Tipo de Feriado'] = 'Feriado Nacional'

df_est = df_os[df_os['impacto_estadual'] == True].groupby(['ano', 'mes'])['tempo_entrega_dias'].mean().to_frame('tmp_feriado').reset_index()
df_est['Tipo de Feriado'] = 'Feriado Estadual'

df_mun = df_os[df_os['impacto_municipal'] == True].groupby(['ano', 'mes'])['tempo_entrega_dias'].mean().to_frame('tmp_feriado').reset_index()
df_mun['Tipo de Feriado'] = 'Feriado Municipal'

df_fac = df_os[df_os['impacto_facultativo'] == True].groupby(['ano', 'mes'])['tempo_entrega_dias'].mean().to_frame('tmp_feriado').reset_index()
df_fac['Tipo de Feriado'] = 'Ponto Facultativo'

# Unificar todos os tipos de feriado em uma única tabela contendo o histórico
analise_final = pd.concat([df_nac, df_est, df_mun, df_fac], ignore_index=True)

# Fixar o tempo médio normal da operação (2 dias)
analise_final['tmp_medio'] = 2.0

# Calcular o Delta (Tempo no feriado menos o tempo normal)
analise_final['delta'] = analise_final['tmp_feriado'] - analise_final['tmp_medio']

# Garantir que o ano e o mês sejam tratados como números inteiros, e não decimais (ex: 2021 em vez de 2021.0)
analise_final['ano'] = analise_final['ano'].astype(int)
analise_final['mes'] = analise_final['mes'].astype(int)

# Reordenar as colunas
analise_final = analise_final[['ano', 'mes', 'Tipo de Feriado', 'tmp_medio', 'tmp_feriado', 'delta']]

# Arredondar os resultados de tempo para 2 casas decimais
analise_final = analise_final.round(2)

# Ordenar cronologicamente para facilitar a visualização
analise_final = analise_final.sort_values(by=['ano', 'mes', 'Tipo de Feriado']).reset_index(drop=True)

print("📊 Prévia dos dados prontos com Ano e Mês separados:")
print(analise_final.head(8))

📊 Prévia dos dados prontos com Ano e Mês separados:
    ano  mes    Tipo de Feriado  tmp_medio  tmp_feriado  delta
0  2021    1   Feriado Estadual        2.0         8.00   6.00
1  2021    1  Feriado Municipal        2.0         3.75   1.75
2  2021    3   Feriado Estadual        2.0         3.50   1.50
3  2021    3  Feriado Municipal        2.0         3.25   1.25
4  2021    3  Ponto Facultativo        2.0         3.00   1.00
5  2021    4  Feriado Municipal        2.0         7.00   5.00
6  2021    4   Feriado Nacional        2.0         7.00   5.00
7  2021    4  Ponto Facultativo        2.0         7.00   5.00


In [23]:
# Salvar o CSV voltando uma pasta (junto com os outros arquivos gerais da sua estrutura)
caminho_saida = '../refined/grafana/os_atrasadas_tipoFeriado_grafana.csv'

analise_final.to_csv(caminho_saida, index=False)
print(f"✅ CSV exportado com sucesso para: {caminho_saida}")

✅ CSV exportado com sucesso para: ../refined/grafana/os_atrasadas_tipoFeriado_grafana.csv
